In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

# 1. Define the dataset folder.
folder = Path(r"C:\FILE PROYEKAN\GCI World 2026\Final Assignment\telecom")

print("Folder contents:")
for p in sorted(folder.iterdir()):
    print(f"- {p.name!r} | suffix={p.suffix!r} | size={p.stat().st_size:,} bytes")

# 2. Locate files by stem only; do not assume the extension.
def find_by_stem(folder: Path, stem: str) -> Path:
    matches = [
        p for p in folder.iterdir()
        if p.is_file() and p.stem.lower() == stem.lower()
    ]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Found {len(matches)} files with stem {stem!r}: {matches}"
        )
    return matches[0]

client_file = find_by_stem(folder, "Client")
record_file = find_by_stem(folder, "Record")

print("\nClient file:", client_file.name, "| suffix:", client_file.suffix)
print("Record file:", record_file.name, "| suffix:", record_file.suffix)

# 3. Load a table without assuming its format.
def load_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix in (".xlsx", ".xls"):
        return pd.read_excel(path)

    # Unknown or missing extension: try CSV, then Excel, and report which worked.
    errors = []

    try:
        df = pd.read_csv(path)
        print(f"[INFO] {path.name}: loaded as CSV (suffix={suffix!r})")
        return df
    except Exception as e:
        errors.append(f"CSV: {e}")

    try:
        df = pd.read_excel(path)
        print(f"[INFO] {path.name}: loaded as Excel (suffix={suffix!r})")
        return df
    except Exception as e:
        errors.append(f"Excel: {e}")

    raise ValueError(f"Failed to load {path}. Errors: {errors}")

client = load_table(client_file)
record = load_table(record_file)

# 4. Inspect each table.
def inspect_df(name: str, df: pd.DataFrame) -> None:
    print(f"\n=== {name} ===")
    print("Shape:", df.shape)

    print("\nData type counts:")
    print(df.dtypes.value_counts())

    print("\nData type per column:")
    print(df.dtypes.to_string())

    missing = df.isna().sum().sort_values(ascending=False)
    print("\nTotal missing:", int(missing.sum()))
    print("Top 20 columns by missing values:")
    print(missing.head(20).to_string())

    print("\nDuplicate rows:", int(df.duplicated().sum()))

    if "Customer_ID" in df.columns:
        print("Customer_ID exists")
        print("  missing:", int(df["Customer_ID"].isna().sum()))
        print("  unique:", df["Customer_ID"].is_unique)
        print("  nunique(dropna=True):", df["Customer_ID"].nunique(dropna=True))
        print("  dtype:", df["Customer_ID"].dtype)
    else:
        print("Customer_ID NOT found")

inspect_df("Client", client)
inspect_df("Record", record)

# 5. Check the relationship on Customer_ID.
print("\n=== CUSTOMER_ID RELATIONSHIP ===")

if "Customer_ID" in client.columns and "Customer_ID" in record.columns:
    c = client["Customer_ID"].dropna()
    r = record["Customer_ID"].dropna()

    if c.dtype != r.dtype:
        print("[WARNING] Customer_ID dtypes differ:", c.dtype, "vs", r.dtype)
        print("Normalizing to stripped strings for set comparison.")
        c_ids = set(c.astype(str).str.strip())
        r_ids = set(r.astype(str).str.strip())
    else:
        c_ids = set(c)
        r_ids = set(r)

    print("Unique IDs in Client:", len(c_ids))
    print("Unique IDs in Record:", len(r_ids))
    print("IDs only in Client:", len(c_ids - r_ids))
    print("IDs only in Record:", len(r_ids - c_ids))
    print("IDs in both:", len(c_ids & r_ids))

    print("Client Customer_ID unique?", client["Customer_ID"].is_unique)
    print("Record Customer_ID unique?", record["Customer_ID"].is_unique)

    try:
        merged = record.merge(
            client,
            on="Customer_ID",
            how="inner",
            validate="one_to_one",
        )
        print("1:1 merge OK. Merged shape:", merged.shape)
    except Exception as e:
        print("1:1 merge failed:", e)
else:
    print("Cannot check relationship: Customer_ID missing in at least one table.")

Folder contents:
- 'Client.csv' | suffix='.csv' | size=17,937,775 bytes
- 'Record.csv' | suffix='.csv' | size=30,182,432 bytes

Client file: Client.csv | suffix: .csv
Record file: Record.csv | suffix: .csv

=== Client ===
Shape: (100000, 50)

Data type counts:
str        21
float64    21
int64       8
Name: count, dtype: int64

Data type per column:
uniqsubs              int64
actvsubs              int64
new_cell                str
crclscod                str
asl_flag                str
totcalls              int64
totmou              float64
totrev              float64
adjrev              float64
adjmou              float64
adjqty                int64
avgrev              float64
avgmou              float64
avgqty              float64
avg3mou               int64
avg3qty               int64
avg3rev               int64
avg6mou             float64
avg6qty             float64
avg6rev             float64
prizm_social_one        str
area                    str
dualband                str
refu